# CWSB microscope detection: train YOLO and get the .pt and .onnx files

Trains a YOLO detection model on the dataset exported from Roboflow and packs the
model files for the device team (Raspberry Pi).

## Where the data comes from

**Default: straight from Roboflow (nothing to upload).** The notebook downloads
dataset version 2 of the project `varsapradaya / cwsb-detection_microscopic`.
It needs your Roboflow **private API key**. The key is never written into this
notebook. Give it in one of two ways:

- **Colab secret (recommended):** click the key icon in the left bar, add a secret
  named `ROBOFLOW_API_KEY` with your key as the value, and turn on "Notebook access".
- **Or type it when asked:** Cell 2 shows a hidden input box if no secret is found.

Your key is in Roboflow under **Settings → API Keys**, and inside the
"Download Dataset → Show download code" snippet. Do not share it or paste it into a cell.

**Other way: a zip from Drive.** Set `SOURCE = 'zip'` in Cell 1, download the version
from Roboflow as a zip (format YOLO26 or YOLOv8) and put it at
`MyDrive/cwsb_data/cwsb_micro_v2_yolo.zip`.

## How to run

`Runtime → Change runtime type → T4 GPU`, then `Runtime → Run all`.
Allow Google Drive when asked (it is only used to keep a copy of the result).
About 15 to 25 minutes. The last cell downloads `cwsb_micro_det_v1.zip`.

## What you get

| File | Use |
|---|---|
| `cwsb_micro_det_v1.pt` | The trained model (Ultralytics) |
| `cwsb_micro_det_v1.onnx` | The same model in ONNX format |
| `cwsb_micro_det_v1_ncnn_model/` | NCNN format, usually the fastest on a Raspberry Pi |
| `classes.txt`, `model_info.json` | Class order, input size, test results |
| `README.txt` | How to run it on the device |
| `test_predictions.jpg`, `results.png` | Pictures for checking |

This is a separate training run from the one inside Roboflow, so its scores will be
close to Roboflow's but not identical. Report the numbers printed by Cell 4.

In [ ]:
# ===== Cell 1: settings =====
SOURCE     = 'roboflow'         # 'roboflow' = download from Roboflow | 'zip' = a zip in Drive
RF_WORKSPACE = 'varsapradaya'
RF_PROJECT   = 'cwsb-detection_microscopic'
RF_VERSION   = 2                # the dataset version in Roboflow
ZIP_PATH   = 'cwsb_data/cwsb_micro_v2_yolo.zip'   # only for SOURCE = 'zip', relative to MyDrive
SAVE_TO_DRIVE = True            # also keep a copy of the result in MyDrive/cwsb_data
VERSION    = 1                  # number used in the output file names
MODEL      = 'yolo26n.pt'       # Nano: the right size for a Raspberry Pi
IMG_SIZE   = 512                # same as the Roboflow version
EPOCHS     = 100
PATIENCE   = 30                 # stop early if there is no improvement for this many epochs
BATCH      = 16
CONF       = 0.25               # confidence used for the test pictures and the ONNX check
SEED       = 42
# The Roboflow version already contains flipped / rotated / brightened copies of the
# training images, so the heavy on-the-fly augmentations are switched off here.
EXTRA_AUGMENT = False

TAG = f'cwsb_micro_det_v{VERSION}'
print('settings ok ->', TAG)

In [ ]:
# ===== Cell 2: install, get the dataset =====
!pip -q install -U ultralytics onnx onnxruntime onnxslim roboflow

import os, glob, json, shutil, zipfile, collections
from pathlib import Path
import yaml, torch

assert SOURCE in ('roboflow', 'zip'), "SOURCE must be 'roboflow' or 'zip'"
DRIVE_DIR = None
if SOURCE == 'zip' or SAVE_TO_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    DRIVE_DIR = Path('/content/drive/MyDrive/cwsb_data')
    DRIVE_DIR.mkdir(parents=True, exist_ok=True)

DATA = Path('/content/data')
shutil.rmtree(DATA, ignore_errors=True)
if SOURCE == 'roboflow':
    from roboflow import Roboflow
    key = None
    try:                                    # Colab secret named ROBOFLOW_API_KEY
        from google.colab import userdata
        key = userdata.get('ROBOFLOW_API_KEY')
    except Exception:
        pass
    if not key:
        from getpass import getpass
        key = getpass('Roboflow private API key (hidden, not saved): ').strip()
    assert key, 'A Roboflow API key is needed. See the first cell.'
    rf = Roboflow(api_key=key)
    del key
    version = rf.workspace(RF_WORKSPACE).project(RF_PROJECT).version(RF_VERSION)
    version.download('yolov8', location=str(DATA), overwrite=True)
    DATA_NAME = f'roboflow {RF_WORKSPACE}/{RF_PROJECT} version {RF_VERSION}'
else:
    ZIP = Path('/content/drive/MyDrive') / ZIP_PATH
    assert ZIP.exists(), f'Not found: {ZIP}. Put the Roboflow zip there or change ZIP_PATH in Cell 1.'
    with zipfile.ZipFile(ZIP) as z:
        z.extractall(DATA)
    DATA_NAME = ZIP.name
print('dataset:', DATA_NAME)

# The Roboflow data.yaml may be one folder down, and its paths are relative. Rewrite it.
src_yaml = sorted(DATA.rglob('data.yaml'), key=lambda p: len(p.parts))[0]
ROOT = src_yaml.parent
cfg = yaml.safe_load(src_yaml.read_text())
NAMES = cfg['names'] if isinstance(cfg['names'], list) else [cfg['names'][k] for k in sorted(cfg['names'])]
splits = {}
for key, folder in (('train', 'train'), ('val', 'valid'), ('test', 'test')):
    d = ROOT / folder / 'images'
    if d.is_dir():
        splits[key] = str(d)
assert 'train' in splits and 'val' in splits, f'train/ and valid/ folders are needed, found: {list(splits)}'
YAML = Path('/content/cwsb_data.yaml')
YAML.write_text(yaml.safe_dump({**splits, 'nc': len(NAMES), 'names': NAMES}, sort_keys=False))

print('classes (in this order):', NAMES)
for key, d in splits.items():
    imgs = [p for p in Path(d).iterdir() if p.suffix.lower() in ('.jpg', '.jpeg', '.png')]
    boxes, empty = collections.Counter(), 0
    for p in imgs:
        lab = Path(d).parent / 'labels' / (p.stem + '.txt')
        lines = [l for l in lab.read_text().splitlines() if l.strip()] if lab.exists() else []
        empty += not lines
        for l in lines:
            boxes[NAMES[int(l.split()[0])]] += 1
    print(f'{key:5s}: {len(imgs):5d} images, {empty:5d} with no object | boxes: {dict(boxes)}')
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE - set the runtime to T4 GPU')

In [ ]:
# ===== Cell 3: train =====
from ultralytics import YOLO

try:
    model = YOLO(MODEL)
except Exception as e:      # older library without this model name
    print(f'{MODEL} could not be loaded ({type(e).__name__}); using yolo11n.pt instead')
    MODEL = 'yolo11n.pt'
    model = YOLO(MODEL)

aug = {} if EXTRA_AUGMENT else dict(mosaic=0.0, mixup=0.0, degrees=0.0, fliplr=0.0, flipud=0.0,
                                     hsv_h=0.0, hsv_s=0.0, hsv_v=0.0, translate=0.0, scale=0.0, erasing=0.0)
model.train(data=str(YAML), imgsz=IMG_SIZE, epochs=EPOCHS, patience=PATIENCE, batch=BATCH,
            seed=SEED, project='/content/runs', name=TAG, exist_ok=True, plots=True, **aug)
RUN = Path('/content/runs') / TAG
BEST = RUN / 'weights' / 'best.pt'
assert BEST.exists(), 'training did not produce best.pt'
print('best model ->', BEST)

In [ ]:
# ===== Cell 4: results on the test set (photos the training never used) =====
best = YOLO(str(BEST))
split = 'test' if 'test' in splits else 'val'
m = best.val(data=str(YAML), split=split, imgsz=IMG_SIZE, conf=0.001, plots=False, verbose=False)

P, R = float(m.box.mp), float(m.box.mr)
RESULT = {'split': split, 'mAP50': round(float(m.box.map50), 4), 'mAP50_95': round(float(m.box.map), 4),
          'precision': round(P, 4), 'recall': round(R, 4), 'per_class': {}}
print(f'Results on the {split} set')
print(f"  mAP@50    : {RESULT['mAP50']:.1%}")
print(f"  precision : {P:.1%}   (of the boxes drawn, how many are right)")
print(f"  recall    : {R:.1%}   (of the real signs, how many are found)")
print('  per class (classes with no object in this set are not listed):')
for i, c in enumerate(m.box.ap_class_index):
    name = NAMES[int(c)]
    RESULT['per_class'][name] = round(float(m.box.ap50[i]), 4)
    print(f'    {name:8s} mAP@50 {float(m.box.ap50[i]):.1%}')
missing = [n for n in NAMES if n not in RESULT['per_class']]
if missing:
    print('  NOT TESTED (no example in this set):', missing)
print('\nThe test set is small, so treat these numbers as rough.')

In [ ]:
# ===== Cell 5: export ONNX and NCNN, and check ONNX against the .pt =====
import numpy as np

onnx_path = Path(best.export(format='onnx', imgsz=IMG_SIZE, opset=12, simplify=True, dynamic=False))
try:
    ncnn_path = Path(YOLO(str(BEST)).export(format='ncnn', imgsz=IMG_SIZE))
except Exception as e:
    ncnn_path = None
    print(f'NCNN export skipped ({type(e).__name__}: {e}). The .pt and .onnx are still fine.')

# Same images through both: the detections must agree.
tdir = Path(splits.get('test', splits['val']))
imgs = sorted(p for p in tdir.iterdir() if p.suffix.lower() in ('.jpg', '.jpeg', '.png'))
labelled = [p for p in imgs if (tdir.parent / 'labels' / (p.stem + '.txt')).exists()
            and (tdir.parent / 'labels' / (p.stem + '.txt')).read_text().strip()]
check = (labelled + [p for p in imgs if p not in labelled])[:24]
onnx_model = YOLO(str(onnx_path), task='detect')
same, worst = 0, 0.0
for p in check:
    a = best.predict(str(p), imgsz=IMG_SIZE, conf=CONF, verbose=False)[0].boxes
    b = onnx_model.predict(str(p), imgsz=IMG_SIZE, conf=CONF, verbose=False)[0].boxes
    ka = sorted(zip(a.cls.tolist(), a.conf.tolist()))
    kb = sorted(zip(b.cls.tolist(), b.conf.tolist()))
    ok = len(ka) == len(kb) and all(x[0] == y[0] for x, y in zip(ka, kb))
    if ok and ka:
        worst = max(worst, max(abs(x[1] - y[1]) for x, y in zip(ka, kb)))
    same += ok
print(f'ONNX check: {same} of {len(check)} images give the same detections as the .pt; '
      f'largest confidence difference {worst:.4f}')
PARITY = {'images': len(check), 'same': same, 'max_conf_diff': round(worst, 5)}

In [ ]:
# ===== Cell 6: pictures of the detections on test images =====
import matplotlib.pyplot as plt

show = check[:12]
fig, ax = plt.subplots(3, 4, figsize=(16, 12))
for a, p in zip(ax.ravel(), show):
    r = best.predict(str(p), imgsz=IMG_SIZE, conf=CONF, verbose=False)[0]
    a.imshow(r.plot()[..., ::-1])
    a.set_title(', '.join(f'{NAMES[int(c)]} {s:.2f}' for c, s in zip(r.boxes.cls, r.boxes.conf)) or 'nothing detected', fontsize=9)
for a in ax.ravel():
    a.axis('off')
plt.tight_layout()
PRED_JPG = Path('/content/test_predictions.jpg')
plt.savefig(PRED_JPG, dpi=90)
plt.show()

In [ ]:
# ===== Cell 7: pack everything and download =====
from google.colab import files

OUT = Path('/content') / TAG
shutil.rmtree(OUT, ignore_errors=True)
OUT.mkdir()
shutil.copy(BEST, OUT / f'{TAG}.pt')
shutil.copy(onnx_path, OUT / f'{TAG}.onnx')
if ncnn_path and ncnn_path.exists():
    shutil.copytree(ncnn_path, OUT / f'{TAG}_ncnn_model')
(OUT / 'classes.txt').write_text('\n'.join(NAMES) + '\n')
for f in ('results.png', 'results.csv', 'confusion_matrix.png'):
    if (RUN / f).exists():
        shutil.copy(RUN / f, OUT / f)
shutil.copy(PRED_JPG, OUT / PRED_JPG.name)
info = {'name': TAG, 'base_model': MODEL, 'image_size': IMG_SIZE, 'classes': NAMES,
        'suggested_confidence': CONF, 'test_results': RESULT, 'onnx_check': PARITY,
        'dataset': DATA_NAME, 'license_note': 'Ultralytics YOLO is AGPL-3.0; check before selling a device'}
(OUT / 'model_info.json').write_text(json.dumps(info, indent=1))
(OUT / 'README.txt').write_text(f"""CWSB microscope detection model: {TAG}

Classes (id: name): {', '.join(f'{i}: {n}' for i, n in enumerate(NAMES))}
Input size: {IMG_SIZE} x {IMG_SIZE}.  Suggested confidence: {CONF} (raise it for fewer false alarms, lower it to miss less).

Run on a Raspberry Pi (Python):
    pip install ultralytics
    from ultralytics import YOLO
    model = YOLO('{TAG}_ncnn_model')        # fastest on a Pi; or '{TAG}.pt' / '{TAG}.onnx'
    r = model.predict('frame.jpg', imgsz={IMG_SIZE}, conf={CONF})[0]
    for c, s, box in zip(r.boxes.cls, r.boxes.conf, r.boxes.xyxy):
        print(model.names[int(c)], float(s), box.tolist())

Use the class NAME for the alarm rule, not the number.
Test results are in model_info.json. They come from a very small test set.
""")
zip_path = shutil.make_archive(str(OUT), 'zip', OUT)
if DRIVE_DIR is not None:
    shutil.copy(zip_path, DRIVE_DIR / f'{TAG}.zip')
    print('saved to Drive ->', DRIVE_DIR / f'{TAG}.zip')
print('contents:', sorted(p.name for p in OUT.iterdir()))
files.download(zip_path)

## After this notebook

- Hand `cwsb_micro_det_v1.zip` to the device team. `README.txt` inside tells them how to run it.
- Report the numbers printed by **Cell 4**, not the ones from Roboflow's own training.
- When more microscope data arrives: label it in Roboflow, generate a new version,
  set `RF_VERSION` to the new version number and `VERSION = 2` in Cell 1, and run again. Compare the Cell 4 numbers.